In [ ]:
# Forensic Fraud Detection Pipeline
# Stage 1: Statistical Investigation → Stage 2: Anomaly Detection → Stage 3: Risk Scoring

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.fraud.data_loader import load, impute, feature_matrix, NUMERIC_FEATURES
from src.fraud.statistical_investigation import (
    univariate_summary,
    benfords_law_test,
    missing_value_heatmap,
    correlation_heatmap,
    label_split_boxplots,
    normality_tests,
)
from src.scoring import compute_risk_score, create_pseudo_labels, compute_percentage_fraud
from src.pipeline import run_full_pipeline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

---
## Stage 1 — Statistical Investigation

We begin with forensic statistics: surfacing suspicious signal in the raw data **without touching the fraud label**. The goal is to understand the shape of the data, identify manipulation patterns, and flag features that warrant closer scrutiny.

In [ ]:
# Load raw data — label column exists but is NOT used until post-hoc evaluation
df_raw = load()
print(f"Dataset: {df_raw.shape[0]:,} transactions × {df_raw.shape[1]} columns")
print(f"Fraud prevalence (held-out): {df_raw['is_fraud'].mean():.1%}")
df_raw.head()

### 1.1 Missing Value Analysis

Correlated missingness across features is a forensic signal — it can indicate that records were selectively tampered with or that a data extraction process dropped fields non-randomly (MNAR).

In [ ]:
print("Missing values per feature:")
missing = df_raw[NUMERIC_FEATURES].isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({"count": missing, "%": missing_pct}).query("count > 0").sort_values("%", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
missing_value_heatmap(df_raw[NUMERIC_FEATURES], ax=ax)
plt.tight_layout()
plt.show()

### 1.2 Univariate Summary

Features with extreme skewness (|skew| > 2), high kurtosis (> 7), or >10% missingness are flagged — these are the distributions most likely to contain manipulated or anomalous records.

In [ ]:
summary = univariate_summary(df_raw, NUMERIC_FEATURES)
summary.style.apply(
    lambda col: ["background-color: #ffe0e0" if v else "" for v in col],
    subset=["forensic_flag"]
)

### 1.3 Benford's Law — Transaction Amount

In naturally occurring financial data, the leading digit of transaction amounts should follow Benford's distribution (digit 1 appears ~30% of the time, digit 9 ~5%). A significant deviation signals potential fabrication or systematic manipulation of amounts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
result = benfords_law_test(df_raw["transaction_amount"], ax=ax)
plt.tight_layout()
plt.show()

print(f"Chi-square statistic : {result['chi2']}")
print(f"p-value              : {result['p_value']}")
print(f"Suspicious           : {result['suspicious']}")

### 1.4 Normality Tests

Most fraud detection anomaly models (Isolation Forest, LOF) are distribution-agnostic, but knowing which features are highly non-normal informs how much we should trust z-score-based outlier rules.

In [ ]:
normality_tests(df_raw, NUMERIC_FEATURES)

### 1.5 Feature Correlation Matrix

Strongly correlated features can double-weight certain signals in an anomaly model. Features that are uncorrelated with everything else are candidates for being the most discriminative fraud signals.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
correlation_heatmap(df_raw, NUMERIC_FEATURES, ax=ax)
plt.tight_layout()
plt.show()

---
## Stage 2 — Anomaly Detection

With a clear picture of the data's statistical properties, we now fit an **Isolation Forest** on the full imputed feature matrix. This is entirely unsupervised — the fraud label is not used.

Isolation Forest works by randomly partitioning the feature space. Anomalous points (potential fraud) are isolated with fewer splits than normal points, producing a lower anomaly score.

In [ ]:
from sklearn.ensemble import IsolationForest

X, _ = feature_matrix(impute(df_raw))

model = IsolationForest(contamination=0.103, random_state=42)
model.fit(X)

raw_scores = model.decision_function(X)
print(f"Score range  : [{raw_scores.min():.4f}, {raw_scores.max():.4f}]")
print(f"Score std    : {raw_scores.std():.4f}")
print(f"Transactions flagged as anomalous: {(model.predict(X) == -1).sum():,}")

In [ ]:
# Distribution of raw anomaly scores
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(raw_scores, bins=60, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(0, color="#DD8452", linewidth=1.5, linestyle="--", label="Decision boundary (0)")
ax.set_xlabel("Isolation Forest decision score")
ax.set_ylabel("Count")
ax.set_title("Anomaly score distribution\n(scores < 0 are flagged as anomalous)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Stage 3 — Risk Scoring

The raw anomaly scores are inverted and normalised to **[0, 1]** to produce an interpretable risk score. We then bucket scores into three tiers calibrated to the ~10% base rate:

| Tier | Label | Threshold | Meaning |
|------|-------|-----------|---------|
| Low | 0 | Bottom 60% | Normal behaviour |
| Medium | 1 | 60th–90th percentile | Worth monitoring |
| High | 2 | Top 10% | Prioritise for review |

In [ ]:
# Run the full pipeline
results = run_full_pipeline(df_raw)

print(f"Estimated fraud rate : {compute_percentage_fraud(df_raw):.1f}%")
print(f"Risk score range     : [{results['risk_score'].min():.4f}, {results['risk_score'].max():.4f}]")

# Tier distribution
labels = create_pseudo_labels(df_raw)
print("\nTier distribution:")
print(labels.value_counts().sort_index().rename({0: "Low (0)", 1: "Medium (1)", 2: "High (2)"}).to_string())

In [ ]:
# Risk score distribution with tier boundaries
scores = results["risk_score"]
p60 = scores.quantile(0.60)
p90 = scores.quantile(0.90)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scores, bins=60, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(p60, color="#f0a500", linewidth=1.5, linestyle="--", label=f"60th pct (low/medium boundary) = {p60:.2f}")
ax.axvline(p90, color="#DD8452", linewidth=1.5, linestyle="--", label=f"90th pct (medium/high boundary) = {p90:.2f}")
ax.set_xlabel("Risk score")
ax.set_ylabel("Count")
ax.set_title("Normalised risk score distribution with tier boundaries")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 highest-risk transactions
print("Top 10 highest-risk transactions:")
results.sort_values("risk_score", ascending=False).head(10)[
    ["transaction_amount", "velocity_score", "distance_from_home",
     "customer_age", "prev_transactions", "risk_score"]
]

---
## Post-hoc Evaluation

The fraud label was held out for the entire pipeline. We now use it **only** to evaluate how well the unsupervised risk score recovers the true fraud signal — this is forensic validation, not model training.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_true = df_raw["is_fraud"]
y_score = results["risk_score"]

auc_roc = roc_auc_score(y_true, y_score)
auc_pr  = average_precision_score(y_true, y_score)

print(f"AUC-ROC  : {auc_roc:.4f}  (0.5 = random, 1.0 = perfect)")
print(f"AUC-PR   : {auc_pr:.4f}   (baseline = {y_true.mean():.3f} = fraud prevalence)")

In [ ]:
# Fraud capture rate by tier — what % of actual frauds land in each tier?
eval_df = df_raw[["is_fraud"]].copy()
eval_df["risk_tier"] = labels.values
eval_df["risk_score"] = results["risk_score"].values

tier_summary = eval_df.groupby("risk_tier").agg(
    transactions=("is_fraud", "count"),
    actual_frauds=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean"),
).rename(index={0: "Low", 1: "Medium", 2: "High"})
tier_summary["fraud_rate"] = tier_summary["fraud_rate"].map("{:.1%}".format)
tier_summary["fraud_capture_%"] = (
    eval_df[eval_df["risk_tier"] == 2]["is_fraud"].sum() / y_true.sum() * 100
)
print("Fraud capture rate in High tier: "
      f"{eval_df[eval_df['risk_tier']==2]['is_fraud'].sum() / y_true.sum():.1%} of all frauds")
tier_summary

In [ ]:
# Risk score distribution split by actual fraud label (post-hoc only)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# KDE by label
for label_val, color, name in [(0, "#4C72B0", "Legitimate"), (1, "#DD8452", "Fraud")]:
    subset = results.loc[df_raw["is_fraud"] == label_val, "risk_score"]
    axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
axes[0].set_xlabel("Risk score")
axes[0].set_ylabel("Density")
axes[0].set_title("Risk score distribution by true label")
axes[0].legend()

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_score)
axes[1].plot(fpr, tpr, color="#4C72B0", lw=2, label=f"AUC-ROC = {auc_roc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve (post-hoc validation)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions split by fraud label — post-hoc reference only
fig = label_split_boxplots(df_raw, NUMERIC_FEATURES)
plt.show()